# Day 2 Submission: Call the Quiet & Bright Twin Finder API
**Team 5 | Swiss Dwellings**

This notebook is the required API caller. For the explained server walkthrough, use [02_development.ipynb](02_development.ipynb). Start the server in a terminal or deploy it to Render before running these cells.

Set `MODEL_URL` to the actual Render URL for online submission evidence. The example is embedded below, so calling the online API needs neither the training CSV, local model files nor a running training notebook. Install `gradio_client==2.7.2` and `requests==2.32.5` in the notebook environment if needed.

In [1]:
import json
import os
import time

import requests
from gradio_client import Client

# Use the actual public Render URL after deployment, or a local terminal server.
MODEL_URL = os.environ.get("MODEL_URL", "http://127.0.0.1:7860").rstrip("/")
EXPECTED_MODEL_SHA256 = "884fb2b59c052b72862f0b07e9ef1b3c983c2d9e60494ae4738d35d2c24e77e3"
print("API URL:", MODEL_URL)
print("Evidence:", "local server" if MODEL_URL.startswith(("http://127.0.0.1", "http://localhost")) else "remote server")

API URL: http://127.0.0.1:7860
Evidence: local server


## 1. Prepare one apartment
Send all 24 measurements in their original units as JSON text. The server applies its saved scaler. Search flags request better acoustics, sunlight and views with comparable area/count. We request three candidates; fewer may qualify.

In [2]:
payload = {
    "label": "Noisy Anomaly Apartment #1 (Street Traffic)",
    "site_id": 3879,
    "building_id": 6051,
    "apartment_id": "2e4d7e256fa899bb4acc18722bceb95f",
    "features": {
        "noise_traffic_day_mean": 31.1931,
        "noise_traffic_day_max": 69.0889,
        "noise_traffic_night_mean": 26.4584,
        "noise_traffic_night_max": 60.6665,
        "window_traffic_day_max": 76.639,
        "window_traffic_night_max": 68.2205,
        "noise_train_day_max": 56.3048,
        "window_train_day_max": 63.8077,
        "sun_equinox_noon_mean": 2.5265,
        "sun_equinox_afternoon_mean": 2.2957,
        "sun_summer_morning_mean": 1.7489,
        "sun_summer_noon_mean": 2.3607,
        "sun_summer_evening_mean": 2.0212,
        "sun_winter_noon_mean": 1.5689,
        "view_sky_mean": 0.0344,
        "view_greenery_mean": 0.0525,
        "view_buildings_mean": 0.005,
        "view_isovist_mean": 14983857.9624,
        "view_streets_mean": 0.0023,
        "apt_total_area": 99.0675,
        "apt_room_count": 9.0,
        "apt_floor_number": 4.0,
        "apt_window_perimeter": 11.4342,
        "apt_closeness_centrality": 0.0372
    }
}
payload.update(top_k=3, improve_acoustics=True, improve_sunlight=True, improve_view=True)
apartment_json = json.dumps(payload)
print("Apartment:", payload["apartment_id"], "| features:", len(payload["features"]))

Apartment: 2e4d7e256fa899bb4acc18722bceb95f | features: 24


## 2. Wake the server and check its model
The free Render service can take about a minute to wake. This bounded retry checks Gradio readiness before creating the API client. `/health` then verifies that the server uses the published model expected by this notebook.

In [3]:
def wake_up(url, attempts=4):
    for attempt in range(1, attempts + 1):
        try:
            requests.get(url + "/config", timeout=45).raise_for_status()
            return
        except requests.RequestException as error:
            print(f"Readiness attempt {attempt}: {type(error).__name__}")
            if attempt < attempts:
                time.sleep(5)
    raise RuntimeError("Server did not become ready. Check the URL and Render logs.")

wake_up(MODEL_URL)
client = Client(MODEL_URL, verbose=False)
server_health = client.predict(api_name="/health")
print(json.dumps(server_health, indent=2))
assert server_health["model_sha256"] == EXPECTED_MODEL_SHA256, "Unexpected server model."

{
  "status": "ready",
  "model_source": "local Day 1 export",
  "model_sha256": "884fb2b59c052b72862f0b07e9ef1b3c983c2d9e60494ae4738d35d2c24e77e3",
  "n_features": 24,
  "latent_dim": 8,
  "catalog_size": 44195
}


## 3. Call predict with gradio_client and print the answer
A score above the threshold is flagged. Scores and threshold ratios are not probabilities or dBA values. The result includes feature errors and up to three real reference apartments; matching IDs are printed for comparison with the HTTP call.

In [4]:
start = time.perf_counter()
client_result = client.predict(apartment_json, api_name="/predict")
print("Called:", MODEL_URL)
print(f"Answered in {time.perf_counter() - start:.2f} seconds")
summary_fields = ["score", "threshold", "is_anomaly", "score_vs_threshold", "matching_candidates"]
print(json.dumps({k: client_result[k] for k in summary_fields}, indent=2))
print("Twin IDs:", [t["apartment_id"] for t in client_result["twins"]])

Called: http://127.0.0.1:7860
Answered in 2.17 seconds
{
  "score": 0.36931124329566956,
  "threshold": 0.179895,
  "is_anomaly": true,
  "score_vs_threshold": 2.0529266699778734,
  "matching_candidates": 9
}
Twin IDs: ['f0bacc174181a13d977427d218651f48', '9500618deba542e170b0af685b48a691', '977e42838c4ddc1773dc0b51ab76eb63']


## 4. Make the same call with ordinary HTTP
POST starts a Gradio job; GET reads its event stream until the result arrives. This shows the request format a later web app can use. The client library hides these two steps.

In [5]:
def call_model(url, apartment_json, timeout=120):
    # First obtain a ticket for the prediction job.
    ticket = requests.post(url + "/gradio_api/call/predict",
                           json={"data": [apartment_json]}, timeout=timeout)
    ticket.raise_for_status()
    event_id = ticket.json()["event_id"]
    # Read complete SSE events; heartbeat messages are not prediction results.
    event = None
    data = []
    with requests.get(f"{url}/gradio_api/call/predict/{event_id}",
                      stream=True, timeout=timeout) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if line.startswith("event:"):
                event = line[6:].strip()
            elif line.startswith("data:"):
                data.append(line[5:].strip())
            elif not line:
                if event == "complete":
                    return json.loads("\n".join(data))[0]
                if event == "error":
                    raise RuntimeError("Model server error: " + " ".join(data))
                event, data = None, []
    raise RuntimeError("The stream ended without a completed prediction.")

http_result = call_model(MODEL_URL, apartment_json)
print("Called:", MODEL_URL)
print(json.dumps({k: http_result[k] for k in summary_fields}, indent=2))
print("Twin IDs:", [t["apartment_id"] for t in http_result["twins"]])
assert abs(http_result["score"] - client_result["score"]) < 1e-5
assert [t["apartment_id"] for t in http_result["twins"]] == [t["apartment_id"] for t in client_result["twins"]]
print("Client and HTTP results agree.")

Called: http://127.0.0.1:7860
{
  "score": 0.36931124329566956,
  "threshold": 0.179895,
  "is_anomaly": true,
  "score_vs_threshold": 2.0529266699778734,
  "matching_candidates": 9
}
Twin IDs: ['f0bacc174181a13d977427d218651f48', '9500618deba542e170b0af685b48a691', '977e42838c4ddc1773dc0b51ab76eb63']
Client and HTTP results agree.


## 5. Interpret the answer
The feature errors show reconstruction mismatches. The twin table uses real measurements, filtered by the requested improvements. A normal result does not certify quietness; an empty twin list means no reference apartments met all goals. Noise-rule metrics are preliminary because of the Day 1 evaluation limitations.

In [6]:
print("Largest feature errors:")
for name, error in sorted(http_result["per_feature_squared_error"].items(), key=lambda p: p[1], reverse=True)[:5]:
    print(f"  {name}: {error:.5f}")
print("\nTwin comparisons:")
for twin in http_result["twins"]:
    f = twin["features"]
    print(twin["apartment_id"], "| daytime traffic:", round(f["noise_traffic_day_max"], 2),
          "| area:", round(f["apt_total_area"], 2), "| area records:", f["apt_room_count"])
print(http_result["search_note"])

Largest feature errors:
  noise_train_day_max: 1.15967
  noise_traffic_day_mean: 1.08155
  apt_window_perimeter: 1.02216
  noise_traffic_night_max: 0.97480
  window_traffic_night_max: 0.88136

Twin comparisons:
f0bacc174181a13d977427d218651f48 | daytime traffic: 49.97 | area: 77.69 | area records: 8.0
9500618deba542e170b0af685b48a691 | daytime traffic: 47.06 | area: 116.87 | area records: 10.0
977e42838c4ddc1773dc0b51ab76eb63 | daytime traffic: 54.66 | area: 90.11 | area records: 8.0
All returned matches pass the requested filters. Fewer matches means fewer eligible apartments.


## Submission evidence
Save this notebook after running it against the actual public Render URL. Printed local URLs establish local readiness only. Add the same public URL to `readme.md` and submit both notebooks with the server source and deployment configuration. No password or token is needed for this public Tuesday API.